In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [17]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1))

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [18]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [19]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('bottle',
   0.2714188020803725,
   tensor(46.0454, grad_fn=<SubBackward0>),
   tensor(-2.3744, grad_fn=<SubBackward0>),
   tensor(-18.8263, grad_fn=<AddBackward0>),
   tensor(24.9319, grad_fn=<AddBackward0>)),
  ('bottle',
   -0.03117353757866148,
   tensor(79.4687, grad_fn=<SubBackward0>),
   tensor(-4.0461, grad_fn=<SubBackward0>),
   tensor(-55.5937, grad_fn=<AddBackward0>),
   tensor(-26.9105, grad_fn=<AddBackward0>)),
  ('pottedplant',
   0.032203955454304545,
   tensor(50.3858, grad_fn=<SubBackward0>),
   tensor(5.4919, grad_fn=<SubBackward0>),
   tensor(1.6613, grad_fn=<AddBackward0>),
   tensor(2.4396, grad_fn=<AddBackward0>)),
  ('pottedplant',
   0.05219813068845269,
   tensor(47.7767, grad_fn=<SubBackward0>),
   tensor(37.1012, grad_fn=<SubBackward0>),
   tensor(45.8112, grad_fn=<AddBackward0>),
   tensor(-29.9506, grad_fn=<AddBackward0>)),
  ('bus',
   -0.1103140813618193,
   tensor(58.4317, grad_fn=<SubBackward0>),
   tensor(52.3405, grad_fn=<SubBackward0>),
   tensor(

In [20]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{'bird': [('bird',
    0.465786070018968,
    tensor(214.3331, grad_fn=<SubBackward0>),
    tensor(199.5273, grad_fn=<SubBackward0>),
    tensor(90.2097, grad_fn=<AddBackward0>),
    tensor(204.0312, grad_fn=<AddBackward0>))]},
 {'sheep': [('sheep',
    0.5369513811736226,
    tensor(17.8386, grad_fn=<SubBackward0>),
    tensor(123.9660, grad_fn=<SubBackward0>),
    tensor(98.8424, grad_fn=<AddBackward0>),
    tensor(145.9522, grad_fn=<AddBackward0>))]},
 {},
 {},
 {'bus': [('bus',
    0.4274339824426079,
    tensor(74.9377, grad_fn=<SubBackward0>),
    tensor(92.1379, grad_fn=<SubBackward0>),
    tensor(151.1437, grad_fn=<AddBackward0>),
    tensor(211.2244, grad_fn=<AddBackward0>))]},
 {},
 {'bus': [('bus',
    0.4331696204996689,
    tensor(196.3992, grad_fn=<SubBackward0>),
    tensor(42.4804, grad_fn=<SubBackward0>),
    tensor(184.1665, grad_fn=<AddBackward0>),
    tensor(17.0346, grad_fn=<AddBackward0>))],
  'horse': [('horse',
    0.4718641231036287,
    tensor(112.3005, grad_

In [37]:
import torch 
nums = torch.arange(10)
nums = nums.sort()[0]

print(nums)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [38]:
nums = [num for num in nums if num == 2 or num == 4 or num == 6]
nums

[tensor(2), tensor(4), tensor(6)]

In [21]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            print(preds)
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]
            print(preds)

    final_preds.append(final_img_preds)

[('bird', 0.465786070018968, tensor(214.3331, grad_fn=<SubBackward0>), tensor(199.5273, grad_fn=<SubBackward0>), tensor(90.2097, grad_fn=<AddBackward0>), tensor(204.0312, grad_fn=<AddBackward0>))]
[]
[('sheep', 0.5369513811736226, tensor(17.8386, grad_fn=<SubBackward0>), tensor(123.9660, grad_fn=<SubBackward0>), tensor(98.8424, grad_fn=<AddBackward0>), tensor(145.9522, grad_fn=<AddBackward0>))]
[]
[('bus', 0.4274339824426079, tensor(74.9377, grad_fn=<SubBackward0>), tensor(92.1379, grad_fn=<SubBackward0>), tensor(151.1437, grad_fn=<AddBackward0>), tensor(211.2244, grad_fn=<AddBackward0>))]
[]
[('bus', 0.4331696204996689, tensor(196.3992, grad_fn=<SubBackward0>), tensor(42.4804, grad_fn=<SubBackward0>), tensor(184.1665, grad_fn=<AddBackward0>), tensor(17.0346, grad_fn=<AddBackward0>))]
[]
[('horse', 0.4718641231036287, tensor(112.3005, grad_fn=<SubBackward0>), tensor(110.4094, grad_fn=<SubBackward0>), tensor(144.5122, grad_fn=<AddBackward0>), tensor(141.5228, grad_fn=<AddBackward0>))]
[

In [16]:
final_preds[8]

{'bicycle': [('bicycle',
   0.5992046695713427,
   tensor(12.8384, grad_fn=<SubBackward0>),
   tensor(124.2831, grad_fn=<SubBackward0>),
   tensor(70.2780, grad_fn=<AddBackward0>),
   tensor(-95.1251, grad_fn=<AddBackward0>)),
  ('bicycle',
   0.5766978866329282,
   tensor(113.3573, grad_fn=<SubBackward0>),
   tensor(55.3489, grad_fn=<SubBackward0>),
   tensor(-33.7235, grad_fn=<AddBackward0>),
   tensor(-47.8872, grad_fn=<AddBackward0>)),
  ('bicycle',
   0.444809103846751,
   tensor(159.5883, grad_fn=<SubBackward0>),
   tensor(46.4134, grad_fn=<SubBackward0>),
   tensor(216.5888, grad_fn=<AddBackward0>),
   tensor(118.8892, grad_fn=<AddBackward0>))],
 'motorbike': [('motorbike',
   0.8779425562780645,
   tensor(112.6371, grad_fn=<SubBackward0>),
   tensor(-27.4791, grad_fn=<SubBackward0>),
   tensor(194.2540, grad_fn=<AddBackward0>),
   tensor(6.1274, grad_fn=<AddBackward0>)),
  ('motorbike',
   0.4606388086452249,
   tensor(218.3090, grad_fn=<SubBackward0>),
   tensor(-11.0427, grad